# Local Qwen3 Analytics RAG

Notebook ini membaca package v2 dari `notebooks/output/latest.json`, menyiapkan data catalog + semantic index, lalu menampilkan widget chat. Qwen3 menghasilkan QueryPlan terstruktur yang langsung dieksekusi terhadap seluruh populasi kandidat. Geometry final selalu diambil dari package analysis dan overlay dirender dengan Matplotlib.

Default: `qwen3:14b`, `qwen3-embedding:0.6b`, dan thinking adaptif. Attempt pertama compact tanpa raw thinking; deep thinking hanya aktif otomatis bila plan awal invalid. Plan terpotong/invalid tidak pernah dieksekusi dan diretry maksimal satu kali.


In [1]:
%matplotlib inline
from pathlib import Path
import sys

BACKEND_ROOT = next(candidate for candidate in (Path.cwd(), Path.cwd().parent) if (candidate / 'explanatory_analysis').is_dir())
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

import pandas as pd
from IPython.display import Markdown, display

from explanatory_analysis.rag import LocalRAG, RAGConfig

rag_config = RAGConfig.default()
rag = LocalRAG(rag_config)
health = rag.health()
display(pd.DataFrame([{"jobId": rag.package.job_id, "package": str(rag.package.path), "chatModel": rag_config.chat_model, "embeddingModel": rag_config.embed_model, "ready": health["ready"], "missingModels": ", ".join(health["missingModels"])}]))
if not health["ready"]:
    raise RuntimeError(f"Model Ollama belum lengkap: {health['missingModels']}")
index_status = rag.prepare_index()
display(Markdown(f"Semantic index siap: **{index_status['documentCount']} evidence cards**, dimensi **{index_status['embeddingDimension']}**."))


,jobId,package,chatModel,embeddingModel,ready,missingModels
0,72ebb6668052,/Users/daf2a/Documents/Challenge/c2/be/path-si...,qwen3:14b,qwen3-embedding:0.6b,True,


Semantic index siap: **45 evidence cards**, dimensi **1024**.

## Tanya data pantry

Jika deep thinking diperlukan, raw thinking ditampilkan sebelum QueryPlan. Dalam jalur compact, UI langsung menampilkan QueryPlan sebagai audit keputusan, lalu hasil executor, evidence, jawaban, dan floorplan overlay. Run baru disimpan ke `llm-rag-v2/runs/`; riwayat `llm-rag-v1` tetap dapat dibuka sebagai legacy tanpa masuk konteks sesi.


In [2]:
chat_widget = rag.widget()
display(chat_widget)


Programmatic API tetap tersedia:

```python
result = rag.ask("Area mana yang paling sering dilewati?")
```
